# EDA — Dataset_0172

**Phase 2 — Exploratory Data Analysis** (Deep Learning course project, Dr. Bahaghighat)

This notebook performs the complete EDA required by the assignment for this dataset:
1. **Visualization**: Histogram, KDE, Boxplot, QQ-Plot, Pair Plot
2. **Descriptive statistics**: mean, median, mode, variance, std, skewness, kurtosis + Kendall confidence intervals
3. **Normality tests**: Shapiro-Wilk, Anderson-Darling, Kolmogorov-Smirnov
4. **Correlation tests**: Pearson, Spearman, Kendall
5. **Outlier detection**: IQR + LOF + Isolation Forest — *flag & report only, NO row is ever removed* (outliers may reflect true physical behaviour of the material)

> An interpretive commentary should be written under each section for the final report.

In [ ]:
# Make the repository root importable
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if (Path.cwd() / '..' / 'src').resolve().exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import get_path
from src.data.loader import load_dataset
from src.eda.visualization import run_all_visualizations
from src.eda.statistics import (descriptive_statistics, normality_tests,
                                correlation_tests, kendall_ci_table)
from src.eda.outliers import outlier_report

DATASET_NAME = 'Dataset_0172'
OUT_DIR = get_path('results_dir') / 'eda' / DATASET_NAME

## 1. Load dataset

The loader drops the `No.` index column and removes constant input features (they carry no predictive information and must not appear in feature-importance analyses).

In [ ]:
bundle = load_dataset(DATASET_NAME)
full = pd.concat([bundle.X, bundle.Y], axis=1)
print('Shape:', full.shape)
print('Inputs :', bundle.feature_names)
print('Outputs:', bundle.output_names)
print('Dropped constant features:', bundle.dropped_constant_features)
full.head()

## 2. Visualization

Histogram + KDE, Boxplot, QQ-Plot, Pair Plot and correlation heatmaps (all figures are also saved to `results/eda/`).

In [ ]:
run_all_visualizations(full, OUT_DIR)
print('Figures saved to', OUT_DIR)

In [ ]:
# Display inline as well
from src.eda.visualization import plot_histograms, plot_boxplots, plot_qq
sns.pairplot(full.select_dtypes('number'), corner=True, plot_kws={'s': 18});

## 3. Descriptive statistics

In [ ]:
desc = descriptive_statistics(full)
desc.to_csv(OUT_DIR / 'descriptive_statistics.csv')
desc

## 4. Normality tests (Shapiro-Wilk, Anderson-Darling, Kolmogorov-Smirnov)

In [ ]:
norm = normality_tests(full)
norm.to_csv(OUT_DIR / 'normality_tests.csv', index=False)
norm

## 5. Correlation tests (Pearson, Spearman, Kendall)

These also give a first answer to research question 1: *is the input-output relationship linear or non-linear?* (compare Pearson vs Spearman/Kendall).

In [ ]:
corr = correlation_tests(full)
corr.to_csv(OUT_DIR / 'correlation_tests.csv', index=False)
corr.sort_values('spearman_rho', key=abs, ascending=False).head(15)

### Kendall confidence intervals (bootstrap)

In [ ]:
kci = kendall_ci_table(full)
kci.to_csv(OUT_DIR / 'kendall_confidence_intervals.csv', index=False)
kci.head(15)

## 6. Outlier detection — IQR + LOF + Isolation Forest

**Important:** samples are only *flagged*; nothing is removed. In friction-processing experiments an apparent outlier can be genuine physical behaviour.

In [ ]:
report = outlier_report(full)
report.to_csv(OUT_DIR / 'outlier_report.csv')
flagged = report[report['n_methods_flagging'] > 0]
print(f'{len(flagged)} of {len(report)} samples flagged by at least one method')
flagged

## 7. Interpretive summary (to complete for the report)

- Which variables deviate from normality, and what does that imply for modelling?
- Which input-output relationships look linear vs non-linear?
- Which samples were flagged as outliers, and is there a plausible physical explanation?
- Any multicollinearity between inputs?
